# Analysis of JV Curves: Experimental Data vs SCAPS Simulations

## Overview
This notebook automates the analysis and comparison of J-V (current density-voltage) curves from laboratory measurements with those simulated using SCAPS (Solar Cell Capacitance Simulator). The goal is to validate the photovoltaic device model by comparing experimental performance with multiple simulation scenarios, identifying which simulations best match the measured behavior through parameter-based and shape-based matching algorithms.

## Section 1: Configuration and Target Parameters

In this section, we define the target performance metrics and experimental calibration parameters. These values represent the expected or desired photovoltaic device performance and are used to:
- **Normalize** the device's electrical characteristics
- **Convert** experimental current to current density
- **Set reference points** for comparing simulations against experiments

In [46]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.interpolate import interp1d
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# -------------------------------------------------------------
# 1. DEFINE YOUR TARGET VALUES HERE (For Plot 3)
# -------------------------------------------------------------
TARGET_VOC = 0.38   # Volts (V)
TARGET_JSC = 20.31    # mA/cm2
TARGET_FF  = 49.56    # %
TARGET_ETA = 4.57     # %

# -------------------------------------------------------------
# 2. EXPERIMENTAL CALIBRATION
# -------------------------------------------------------------
# Define your cell's active area in cm² to convert I to J (mA/cm²)
ACTIVE_AREA_CM2 = 0.1 

# SCAPS typically outputs negative photocurrent (e.g., -25mA/cm2).
# If your experimental measurement yielded positive current values, 
# the code automatically inverts them to make both curves comparable.
INVERT_EXP_CURRENT = True


## Theoretical Foundation: The Single-Diode Model

### Overview
The behavior of photovoltaic devices under illumination is commonly modeled using the **single-diode equivalent circuit**, which provides a physical interpretation of J-V curves. This model connects microscopic device physics to macroscopic electrical measurements.

### Mathematical Framework

The current-voltage relationship in solar cells is described by the **Shockley ideal diode equation**:

$$J = J_L - J_0 \left[\exp\left(\frac{qV}{nk_BT}\right) - 1\right] - \frac{V}{R_{sh}}$$

where:
- $J$ = current density (mA/cm²)
- $J_L$ = light-generated current density (photocurrent)
- $J_0$ = dark saturation current density (reverse saturation)
- $q$ = elementary charge (1.602 × 10⁻¹⁹ C)
- $V$ = applied voltage (V)
- $n$ = ideality factor (typically 1-2, accounts for recombination mechanisms)
- $k_B$ = Boltzmann constant (1.381 × 10⁻²³ J/K)
- $T$ = absolute temperature (K, typically 298 K at STC)
- $R_{sh}$ = shunt resistance (parallel resistance, Ω·cm²)

### Physical Interpretation

1. **First Term $(J_L)$**: Represents photocurrent generated by absorbed photons. Independent of voltage.

2. **Second Term $(J_0[\exp(...)-1])$**: Describes dark current from thermal generation and recombination. Exponentially increases with voltage above thermal voltage $V_T = \frac{k_BT}{q} \approx 26$ mV at 298 K.

3. **Third Term $(V/R_{sh})$**: Leakage current due to shunt paths; becomes significant at high voltages or low light conditions.

### Under Illumination: Modified Equation

Under illumination, the net current is:

$$J = J_L - J_0 \left[\exp\left(\frac{q(V + J \cdot R_s)}{nk_BT}\right) - 1\right] - \frac{V + J \cdot R_s}{R_{sh}}$$

This includes **series resistance** $R_s$ (resistance of the contacts, semiconductor, and grid), which causes an additional voltage drop proportional to current.

### Key Performance Metrics

At the light-dark current crossover:

$$\boxed{\text{At } V_{oc}: J = 0 \implies J_L = J_0 \left[\exp\left(\frac{qV_{oc}}{nk_BT}\right) - 1\right]}$$

The **Fill Factor** represents the ratio of the maximum power rectangle to the theoretical maximum:

$$FF = \frac{P_{max}}{V_{oc} \cdot J_{sc}} = \frac{V_{mp} \cdot J_{mp}}{V_{oc} \cdot J_{sc}}$$

where $(V_{mp}, J_{mp})$ is the operating point of maximum power output.

**Efficiency** (Power Conversion Efficiency):

$$\eta = \frac{P_{max}}{P_{in}} = \frac{V_{oc} \cdot J_{sc} \cdot FF}{P_{in}}$$

where $P_{in}$ is incident solar power (typically 100 mW/cm² under STC).

### How Series and Shunt Resistance Affect Curve Shape

- **High $R_s$**: Reduces FF, causes "S-shape" at high voltages; limits maximum power
- **Low $R_{sh}$**: Reduces $V_{oc}$, increases leakage at forward bias; evident as steep slope near $J_{sc}$
- **Ideal case**: $R_s \to 0$, $R_{sh} \to \infty$, resulting in sharp rectangular J-V curve

### SCAPS and the Single-Diode Model

SCAPS solves the complete semiconductor transport equations without assuming the single-diode form. However, by analyzing the resulting J-V curves, SCAPS-simulated data can be fitted to extract effective values of $J_L$, $J_0$, $n$, $R_s$, and $R_{sh}$, providing a connection between microscopic device parameters (layer thickness, doping, defects) and macroscopic circuit behavior.

## Section 2: Experimental Data Reading and Visualization

### Background: The J-V Curve
The current density-voltage (J-V) curve is a fundamental characterization of photovoltaic devices. It shows how the current output of a solar cell varies with applied voltage under steady illumination. Key parameters extracted from the J-V curve include:

- **$V_{oc}$ (Open Circuit Voltage)**: Maximum voltage output when no current flows  
- **$J_{sc}$ (Short Circuit Current Density)**: Maximum current when voltage is zero  
- **$FF$ (Fill Factor)**: Ratio of maximum power to the product of $V_{oc}$ and $J_{sc}$  
- **$\eta$ (Power Conversion Efficiency)**: Percentage of incident solar power converted to electrical power

In this section, we read experimental J-V data from your laboratory measurements and prepare it for comparison with simulations.

In [47]:
# Read the experimental file
exp_file = "data/CdS3_IV Graph_1.xlsx"
exp_df_raw = pd.read_excel(exp_file)

# Find the target column and its respective Voltage / Current data
col_name = "CdS3_6 2026-01-22 14-24-51"
col_idx = exp_df_raw.columns.get_loc(col_name)

# Skip the 1st row (which contains the string headers 'Vmeas' and 'Imeas')
v_exp = exp_df_raw.iloc[1:, col_idx].astype(float).values
i_exp = exp_df_raw.iloc[1:, col_idx + 1].astype(float).values

# Convert Current (A) to Current Density (mA/cm2)
j_exp = (i_exp * 1000) / ACTIVE_AREA_CM2
if INVERT_EXP_CURRENT:
    j_exp = -j_exp

exp_data = pd.DataFrame({'v(V)': v_exp, 'jtot(mA/cm2)': j_exp}).dropna()

# Store experimental voltage and current density limits for filtering simulations
EXP_V_MIN = exp_data['v(V)'].min()
EXP_V_MAX = exp_data['v(V)'].max()
EXP_J_MIN = exp_data['jtot(mA/cm2)'].min()
EXP_J_MAX = exp_data['jtot(mA/cm2)'].max()

# ------------------ PLOT 1: EXPERIMENTAL ------------------
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=exp_data['v(V)'], 
    y=exp_data['jtot(mA/cm2)'], 
    mode='lines', name='Experimental', 
    line=dict(color='black', dash='dash', width=3)
))
fig1.update_layout(
    title='Plot 1: Experimental JV Curve',
    xaxis_title='Voltage (V)',
    yaxis_title='Current Density (mA/cm2)',
    template='plotly_white', width=900, height=600
)
fig1.show()

## Section 3: SCAPS Simulations Overview

### What is SCAPS?
**SCAPS** (Solar Cell Capacitance Simulator) is a specialized software tool developed at Ghent University for simulating the electrical behavior of thin-film photovoltaic devices. It solves Poisson's equation and the continuity equations in a multi-layer semiconductor structure to predict device performance under standard test conditions (STC: 1000 W/m², AM1.5 spectrum, 25°C).

### SCAPS Simulation Methodology
SCAPS performs detailed numerical simulations of solar cells by:

1. **Device Architecture Definition**: Specifies material layers, thickness, doping levels, and defect parameters
2. **Physical Parameters**: Sets electron and hole transport properties, recombination mechanisms, and optical characteristics
3. **Numerical Solution**: Solves coupled differential equations:
   - **Poisson Equation**: $\frac{d^2\phi}{dx^2} = -\frac{q}{\varepsilon_0\varepsilon_r}(p - n + N_d^+ - N_a^-)$
   - **Continuity Equations**: 
     - Electrons: $\frac{\partial n}{\partial t} + \frac{1}{q}\frac{\partial J_n}{\partial x} = G - R$
     - Holes: $\frac{\partial p}{\partial t} - \frac{1}{q}\frac{\partial J_p}{\partial x} = G - R$

   where $\phi$ is electrostatic potential, $q$ is elementary charge, $n$ and $p$ are electron and hole concentrations, $J_n$ and $J_p$ are current densities, $G$ is generation rate, and $R$ is recombination rate.

4. **I-V Characteristics**: Calculates the J-V curve by sweeping applied voltage and computing resulting current

### Key Physical Processes in SCAPS
- **Photocurrent Generation**: Incident photons with energy > bandgap create electron-hole pairs
- **Carrier Transport**: Electrons and holes drift under electric field and diffuse down concentration gradients
- **Recombination**: Losses from multiple recombination mechanisms (radiative, Auger, defect-assisted SRH)
- **Interface Effects**: Band alignment, interfacial defects, and charge extraction barriers

### Output from SCAPS
SCAPS generates J-V curves showing current density (mA/cm²) as a function of applied voltage (V), along with derived parameters: $V_{oc}$, $J_{sc}$, and efficiency $\eta$. Multiple simulations with varied parameters (e.g., thickness, doping, defect density) help identify optimal device configurations.

In [48]:
# Read the SCAPS file
sim_file = "data/CdS3_IV_scaps.xlsx"
sim_df_raw = pd.read_excel(sim_file, header=None)

# Find where the parameters for each simulation start
batch_indices = sim_df_raw[sim_df_raw.apply(lambda row: row.astype(str).str.contains("Batch parameters").any(), axis=1)].index.tolist()

scaps_data = []

for idx in batch_indices:
    # 1. Extract Name/Identification right below Batch Parameters
    param_name = str(sim_df_raw.iloc[idx + 1, 0]).strip()
    param_val = str(sim_df_raw.iloc[idx + 1, 1]).strip()
    label = f"{param_name} {param_val}"
    
    # 2. Extract IV Curves (Searching from the Voltage section until Voc is found)
    iv_rows = []
    for j in range(idx + 5, min(idx + 80, len(sim_df_raw))):
        row0 = str(sim_df_raw.iloc[j, 0]).strip()
        row1 = str(sim_df_raw.iloc[j, 1]).strip()
        if pd.isna(sim_df_raw.iloc[j, 0]) or "Voc" in row0:
            if "Voc" in row0: break
            continue # Skip empty rows
        try:
            iv_rows.append([float(row0), float(row1)])
        except ValueError:
            pass
            
    iv_df = pd.DataFrame(iv_rows, columns=['v(V)', 'jtot(mA/cm2)'])
    
    # 3. Extract Performance Parameters (Voc, Jsc, FF, eta)
    metrics_block = sim_df_raw.iloc[idx : min(idx + 80, len(sim_df_raw))]
    voc = jsc = ff = eta = np.nan
    for _, row in metrics_block.iterrows():
        col0 = str(row[0]).strip()
        val1 = str(row[1]).strip()
        if "Voc =" in col0: voc = float(val1)
        elif "Jsc =" in col0: jsc = float(val1)
        elif "FF =" in col0: ff = float(val1)
        elif "eta =" in col0: eta = float(val1)
            
    scaps_data.append({
        'label': label,
        'v(V)': iv_df['v(V)'].values,
        'jtot(mA/cm2)': iv_df['jtot(mA/cm2)'].values,
        'Voc': voc, 'Jsc': jsc, 'FF': ff, 'eta': eta
    })

# ------------------ RESULTS TABLE ------------------
print("\n=== EXTRACTED SCAPS PARAMETERS ===")
summary_df = pd.DataFrame([{
    'ID / Simulation': d['label'], 'Voc (V)': d['Voc'], 
    'Jsc (mA/cm2)': d['Jsc'], 'FF (%)': d['FF'], 'eta (%)': d['eta']
} for d in scaps_data])
display(summary_df)

# ------------------ PLOT 2: ALL SIMULATIONS ------------------
fig2 = go.Figure()
for i, d in enumerate(scaps_data):
    # Filter to experimental maximum limits
    mask = (d['v(V)'] <= EXP_V_MAX) & (d['jtot(mA/cm2)'] <= EXP_J_MAX)
    v_filtered = d['v(V)'][mask]
    j_filtered = d['jtot(mA/cm2)'][mask]
    
    # Text to display when hovering over the data points
    hover_text = f"<b>{d['label']}</b><br>Voc: {d['Voc']} V<br>Jsc: {d['Jsc']} mA/cm2<br>FF: {d['FF']}%<br>eta: {d['eta']}%"
    
    # Name displayed in the legend (ID + Voc and eta)
    legend_name = f"Simulation {i+1} (Voc: {d['Voc']}V, eta: {d['eta']}%)"
    
    fig2.add_trace(go.Scatter(
        x=v_filtered, y=j_filtered, mode='lines', 
        name=legend_name, hoverinfo='text', hovertext=hover_text
    ))

fig2.update_layout(
    title='Plot 2: All SCAPS Simulation IV Curves',
    xaxis_title='Voltage (V)', yaxis_title='Current Density (mA/cm2)',
    hovermode='closest', template='plotly_white', width=1000, height=600
)
fig2.show()


=== EXTRACTED SCAPS PARAMETERS ===


,ID / Simulation,Voc (V),Jsc (mA/cm2),FF (%),eta (%)
0,CdS (L2)>>thickness[µm]: 0.05,0.422144,25.469707,49.8801,5.3631
1,CdS (L2)>>thickness[µm]: 0.06,0.422452,25.144599,50.0830,5.3200
2,CdS (L2)>>thickness[µm]: 0.07,0.422730,24.838661,50.4528,5.2976
3,CdS (L2)>>thickness[µm]: 0.08,0.422987,24.553663,50.9140,5.2879
4,CdS (L2)>>thickness[µm]: 0.09,0.423226,24.290813,51.4201,5.2862
5,CdS (L2)>>thickness[µm]: 0.1,0.423450,24.050558,51.9419,5.2899
6,CdS (L2)>>thickness[µm]: 0.11,0.423662,23.832609,52.4609,5.2970
7,CdS (L2)>>thickness[µm]: 0.12,0.423861,23.636089,52.9654,5.3063
8,CdS (L2)>>thickness[µm]: 0.12,0.423861,23.636089,52.9654,5.3063


## Section 4: Parameter-Based Comparison with Target Values

### Methodology
In this section, we compare simulations against **target performance parameters** using a scoring algorithm. For each simulation, we calculate a normalized distance metric:

$$\text{Score} = \sum_{i} \left(\frac{X_i^{sim} - X_i^{target}}{X_i^{target}}\right)^2$$

where $X_i$ represents the performance parameters:
- $V_{oc}$ (Open Circuit Voltage)
- $J_{sc}$ (Short Circuit Current Density)  
- $FF$ (Fill Factor)
- $\eta$ (Power Conversion Efficiency)

**Interpretation**: Lower scores indicate simulations whose parameters are closer to the target values. This approach is useful for identifying device designs that achieve desired specifications.

### Application
The top 5 simulations with the lowest parameter scores are plotted alongside experimental data to evaluate whether parameter matching correlates with realistic device behavior.

In [49]:
# Calculate a Score (the lower the score, the closer to the target parameters)
for d in scaps_data:
    score = 0
    score += ((d['Voc'] - TARGET_VOC) / TARGET_VOC)**2
    score += ((d['Jsc'] - TARGET_JSC) / TARGET_JSC)**2
    score += ((d['FF'] - TARGET_FF) / TARGET_FF)**2
    score += ((d['eta'] - TARGET_ETA) / TARGET_ETA)**2
    d['param_score'] = score

top_5_params = sorted(scaps_data, key=lambda x: x['param_score'])[:5]

# ------------------ PLOT 3: TOP 5 VS EXPERIMENTAL (PARAMETERS) ------------------
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=exp_data['v(V)'], y=exp_data['jtot(mA/cm2)'], 
    mode='lines', name='Experimental', line=dict(color='black', dash='dash', width=3)
))

for i, d in enumerate(top_5_params):
    # Filter to experimental maximum limits
    mask = (d['v(V)'] <= EXP_V_MAX) & (d['jtot(mA/cm2)'] <= EXP_J_MAX)
    v_filtered = d['v(V)'][mask]
    j_filtered = d['jtot(mA/cm2)'][mask]
    
    hover_text = f"<b>{d['label']}</b><br>Voc: {d['Voc']}<br>eta: {d['eta']}"
    fig3.add_trace(go.Scatter(
        x=v_filtered, y=j_filtered, mode='lines', 
        name=f"Top {i+1} (Parameter Score)", hoverinfo='text', hovertext=hover_text
    ))

fig3.update_layout(
    title='Plot 3: Top 5 Simulations Closest to Target Parameters',
    xaxis_title='Voltage (V)', yaxis_title='Current Density (mA/cm2)',
    hovermode='closest', template='plotly_white', width=900, height=600
)
fig3.show()

# Create comparison table
print("\n" + "="*120)
print("TOP 5 PARAMETER MATCHES: EXPERIMENTAL VS SIMULATIONS")
print("="*120)
print(f"\n{'Metric':<20} {'Experimental':<18} {'Top 1':<18} {'Top 2':<18} {'Top 3':<18} {'Top 4':<18} {'Top 5':<18}")
print("-"*120)
print(f"{'Voc (V)':<20} {TARGET_VOC:<18.4f} {top_5_params[0]['Voc']:<18.4f} {top_5_params[1]['Voc']:<18.4f} {top_5_params[2]['Voc']:<18.4f} {top_5_params[3]['Voc']:<18.4f} {top_5_params[4]['Voc']:<18.4f}")
print(f"{'Jsc (mA/cm²)':<20} {TARGET_JSC:<18.4f} {top_5_params[0]['Jsc']:<18.4f} {top_5_params[1]['Jsc']:<18.4f} {top_5_params[2]['Jsc']:<18.4f} {top_5_params[3]['Jsc']:<18.4f} {top_5_params[4]['Jsc']:<18.4f}")
print(f"{'FF (%)':<20} {TARGET_FF:<18.2f} {top_5_params[0]['FF']:<18.2f} {top_5_params[1]['FF']:<18.2f} {top_5_params[2]['FF']:<18.2f} {top_5_params[3]['FF']:<18.2f} {top_5_params[4]['FF']:<18.2f}")
print(f"{'η (%)':<20} {TARGET_ETA:<18.4f} {top_5_params[0]['eta']:<18.4f} {top_5_params[1]['eta']:<18.4f} {top_5_params[2]['eta']:<18.4f} {top_5_params[3]['eta']:<18.4f} {top_5_params[4]['eta']:<18.4f}")
print(f"{'Param Score':<20} {'---':<18} {top_5_params[0]['param_score']:<18.6f} {top_5_params[1]['param_score']:<18.6f} {top_5_params[2]['param_score']:<18.6f} {top_5_params[3]['param_score']:<18.6f} {top_5_params[4]['param_score']:<18.6f}")
print("="*120)


TOP 5 PARAMETER MATCHES: EXPERIMENTAL VS SIMULATIONS

Metric               Experimental       Top 1              Top 2              Top 3              Top 4              Top 5             
------------------------------------------------------------------------------------------------------------------------
Voc (V)              0.3800             0.4239             0.4239             0.4237             0.4234             0.4232            
Jsc (mA/cm²)         20.3100            23.6361            23.6361            23.8326            24.0506            24.2908           
FF (%)               49.56              52.97              52.97              52.46              51.94              51.42             
η (%)                4.5700             5.3063             5.3063             5.2970             5.2899             5.2862            
Param Score          ---                0.070822           0.070822           0.072017           0.074119           0.077326          


## Section 5: Shape-Based Comparison Using Mean Squared Error (MSE)

### Motivation
While parameter-based matching identifies designs meeting target specifications, it does not guarantee realistic device physics. Two J-V curves can have identical parameters but very different shapes, indicating different dominant loss mechanisms. This section employs a **shape-matching algorithm** to identify simulations whose curve morphology most closely resembles the experimental measurement.

### Mathematical Approach
For each simulation, we:

1. **Interpolate** the simulated J-V curve to the exact voltages measured in the experiment
2. **Calculate Mean Squared Error** in the overlapping voltage range:

$$\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} \left(J_i^{exp} - J_i^{sim}(V_i)\right)^2$$

where $J^{exp}$ is experimental current density at measured voltages, $J^{sim}(V_i)$ is interpolated simulated current density at those same voltages, and $N$ is the number of measurement points.

3. **Identify** the top 5 simulations with **lowest MSE** values

### Interpretation
Lower MSE values indicate better agreement in curve shape. This metric is sensitive to:
- The "knee" or transition region of the J-V curve (determined by series/shunt resistance and ideality factors)
- Overall current deficit and voltage losses
- Non-ideal behavior like S-shaped curves or kinks caused by interface barriers

### Comparison with Parameter-Based Matching
- **Parameter-based**: Focuses on specific values ($V_{oc}$, $J_{sc}$, $FF$, $\eta$)
- **Shape-based (MSE)**: Focuses on the entire curve morphology and transition characteristics

Together, these approaches provide complementary insights into which simulations best capture experimental device behavior.

In [50]:
for d in scaps_data:
    v_sim = d['v(V)']
    j_sim = d['jtot(mA/cm2)']
    
    # Create an interpolator to compare with the exact voltages measured in the lab
    interp_func = interp1d(v_sim, j_sim, kind='linear', fill_value="extrapolate")
    
    # Evaluate only in the overlapping region (common Voltage Range)
    v_min, v_max = max(v_sim.min(), exp_data['v(V)'].min()), min(v_sim.max(), exp_data['v(V)'].max())
    valid_exp = exp_data[(exp_data['v(V)'] >= v_min) & (exp_data['v(V)'] <= v_max)]
    
    if len(valid_exp) > 5:
        j_sim_interp = interp_func(valid_exp['v(V)'])
        mse = mean_squared_error(valid_exp['jtot(mA/cm2)'], j_sim_interp)
        d['mse'] = mse
    else:
        d['mse'] = float('inf')

top_5_mse = sorted(scaps_data, key=lambda x: x.get('mse', float('inf')))[:5]

# ------------------ PLOT 4: TOP 5 VS EXPERIMENTAL (MSE SHAPE) ------------------
fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=exp_data['v(V)'], y=exp_data['jtot(mA/cm2)'], 
    mode='lines', name='Experimental', line=dict(color='black', dash='dash', width=3)
))

for i, d in enumerate(top_5_mse):
    # Filter to experimental maximum limits
    mask = (d['v(V)'] <= EXP_V_MAX) & (d['jtot(mA/cm2)'] <= EXP_J_MAX)
    v_filtered = d['v(V)'][mask]
    j_filtered = d['jtot(mA/cm2)'][mask]
    
    hover_text = f"<b>{d['label']}</b><br>MSE: {d['mse']:.4f}"
    fig4.add_trace(go.Scatter(
        x=v_filtered, y=j_filtered, mode='lines', 
        name=f"Top {i+1} MSE: {d['label']}", hoverinfo='text', hovertext=hover_text
    ))

fig4.update_layout(
    title='Plot 4: Top 5 Simulations with the Most Similar Curve Shape (MSE)',
    xaxis_title='Voltage (V)', yaxis_title='Current Density (mA/cm2)',
    hovermode='closest', template='plotly_white', width=900, height=600
)
fig4.show()

## Section 6: Summary Comparison - Best Matches

This section presents a side-by-side comparison of the experimental J-V curve with the best matching simulations from both approaches:
- **Best Shape Match (MSE)**: Simulation with the most similar curve morphology to experimental data
- **Best Parameter Match**: Simulation with performance parameters (Voc, Jsc, FF, η) closest to targets

In [51]:
# Extract best matches
best_mse = top_5_mse[0]  # Best shape match
best_params = top_5_params[0]  # Best parameter match

# Create comparison figure
fig_summary = go.Figure()

# Add experimental data
fig_summary.add_trace(go.Scatter(
    x=exp_data['v(V)'], y=exp_data['jtot(mA/cm2)'], 
    mode='lines', name='Experimental', 
    line=dict(color='black', dash='dash', width=3),
    hovertemplate='<b>Experimental</b><br>V: %{x:.3f} V<br>J: %{y:.2f} mA/cm²<extra></extra>'
))

# Add best shape match
mask_mse = (best_mse['v(V)'] <= EXP_V_MAX) & (best_mse['jtot(mA/cm2)'] <= EXP_J_MAX)
v_mse = best_mse['v(V)'][mask_mse]
j_mse = best_mse['jtot(mA/cm2)'][mask_mse]
fig_summary.add_trace(go.Scatter(
    x=v_mse, y=j_mse, 
    mode='lines', name=f'Best Shape Match (MSE: {best_mse["mse"]:.4f})',
    line=dict(color='blue', width=2),
    hovertemplate='<b>Best Shape Match</b><br>V: %{x:.3f} V<br>J: %{y:.2f} mA/cm²<extra></extra>'
))

# Add best parameter match
mask_params = (best_params['v(V)'] <= EXP_V_MAX) & (best_params['jtot(mA/cm2)'] <= EXP_J_MAX)
v_params = best_params['v(V)'][mask_params]
j_params = best_params['jtot(mA/cm2)'][mask_params]
fig_summary.add_trace(go.Scatter(
    x=v_params, y=j_params, 
    mode='lines', name=f'Best Parameter Match',
    line=dict(color='red', width=2),
    hovertemplate='<b>Best Parameter Match</b><br>V: %{x:.3f} V<br>J: %{y:.2f} mA/cm²<extra></extra>'
))

fig_summary.update_layout(
    title='Summary: Experimental vs Best Matching Simulations',
    xaxis_title='Voltage (V)',
    yaxis_title='Current Density (mA/cm²)',
    hovermode='closest',
    template='plotly_white',
    width=1000,
    height=600
)
fig_summary.show()

# Create comparison table
print("\n" + "="*100)
print("BEST MATCHES SUMMARY")
print("="*100)
print(f"\n{'Metric':<20} {'Experimental':<20} {'Best Shape (MSE)':<25} {'Best Parameters':<25}")
print("-"*100)
print(f"{'Voc (V)':<20} {TARGET_VOC:<20.4f} {best_mse['Voc']:<25.4f} {best_params['Voc']:<25.4f}")
print(f"{'Jsc (mA/cm²)':<20} {TARGET_JSC:<20.4f} {best_mse['Jsc']:<25.4f} {best_params['Jsc']:<25.4f}")
print(f"{'FF (%)':<20} {TARGET_FF:<20.2f} {best_mse['FF']:<25.2f} {best_params['FF']:<25.2f}")
print(f"{'η (%)':<20} {TARGET_ETA:<20.4f} {best_mse['eta']:<25.4f} {best_params['eta']:<25.4f}")
print(f"{'MSE':<20} {'---':<20} {best_mse['mse']:<25.6f} {best_params.get('mse', '---'):<25}")
print(f"{'Param Score':<20} {'---':<20} {best_mse.get('param_score', '---'):<25} {best_params['param_score']:<25.6f}")
print("="*100)


BEST MATCHES SUMMARY

Metric               Experimental         Best Shape (MSE)          Best Parameters          
----------------------------------------------------------------------------------------------------
Voc (V)              0.3800               0.4239                    0.4239                   
Jsc (mA/cm²)         20.3100              23.6361                   23.6361                  
FF (%)               49.56                52.97                     52.97                    
η (%)                4.5700               5.3063                    5.3063                   
MSE                  ---                  317.841866                317.84186566311104       
Param Score          ---                  0.0708217394631384        0.070822                 
